# 21 — G2: farklı kaynaklarda geçici model karşılaştırması

**Donanım: CPU. GPU gerekmez.** Eğitim yapılmaz; Notebook 20'nin modelleri kullanılır.

1. Colab'da bu notebook'u yükleyin ve **Tümünü çalıştır** seçin.
2. İstendiğinde `calorie_g2_pose_model_suite_report.zip` dosyanızı yükleyin.
3. Sonuç: `calorie_g2_creator_comparison_report.zip`.

V6-C ve G2 modelleri, eğitim/iç doğrulama/iç test bölümlerinden dışlanmış üç video grubunda aynı gate ile karşılaştırılır. Bunlar Notebook 17'de kullanılmış kalibrasyon kaynaklarıdır; yeni, kilitli bir test değildir. Eski etiket inceleme uyarıları ve eksik sınıflar nedeniyle sonuç **geçicidir**. Bu notebook model yükseltmez, eşik veya ensemble ağırlığı seçmez.

Ensemble skoru bu kez gerçek ortalama olasılıklardan hesaplanır. Notebook 20'nin hatalı iç validation alanını düzeltmez; bu ayrı veri grubunda yeni bir ölçümdür.


In [ ]:
%pip install -q torch numpy pandas==2.2.3 scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import json
import shutil
import urllib.request
import zipfile
import hashlib
import os
import time
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, recall_score,
)
from torch import nn

SEED = 42
WINDOW = 64
STRIDE = 32
RETIRED_CLASSES = {"plank"}
CALIBRATION_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}
np.random.seed(SEED)
torch.manual_seed(SEED)

# CPU is deliberate: inference and threshold sweeps are small and deterministic.
DEVICE = torch.device("cpu")
WORK_ROOT = Path.cwd() / "calorie_nb21_work"
DOWNLOAD_ROOT = WORK_ROOT / "downloads"
EXTRACT_ROOT = WORK_ROOT / "extracted"
REPORT_ROOT = WORK_ROOT / "calorie_g2_creator_comparison_report"
for path in [DOWNLOAD_ROOT, EXTRACT_ROOT, REPORT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
print("Work root:", WORK_ROOT.resolve())

torch.set_num_threads(2)
FROZEN_GATE_THRESHOLD = 0.675
EXPECTED_HASHES = {'v6c_report': 'd2e8263c04954f13157ca76221de5d408b04b1ec', 'calibration_landmarks': '948db2dece517a6afa0a8cefd789381222db9a76', 'g2_report': 'c2bcb5b01c65872ad6b990b0eb20639d177dc86281a07abcd270710dd366aea7'}

# These creators were excluded from the V6/G2 splits, but were already used
# in Notebook 17 calibration. They are NOT a new locked test.
EVIDENCE_STATUS = "provisional_existing_creator_calibration_not_locked_test"
print("CPU inference only; no training. Labels still require visual review.")


## Sabit kaynakları indir ve Notebook 20 raporunu yükle


In [ ]:
PINNED_COMMIT = "fb7fea4c7fc4f3d3d01df2a474d49460b3a2df73"
RAW = f"https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/ml/artifacts"
ASSETS = {
    "v6c_report": f"{RAW}/v6c/model/calorie_v6c_kinematic_features_tcn_report.zip",
    "calibration_landmarks": f"{RAW}/v2/landmarks/calorie_youtube_mixed_landmarks_v3.zip",
}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def asset_digest(path):
    data = Path(path).read_bytes()
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()

def checked_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

local_repo = os.environ.get("CALORIE_LOCAL_REPO")  # optional local smoke-test input
for name, url in ASSETS.items():
    destination = DOWNLOAD_ROOT / Path(url).name
    if local_repo:
        relative = url.split("/ml/artifacts/", 1)[1]
        shutil.copyfile(Path(local_repo) / "ml/artifacts" / relative, destination)
    elif not destination.exists() or asset_digest(destination) != EXPECTED_HASHES[name]:
        request = urllib.request.Request(url, headers={"User-Agent": "cALorie-Notebook21"})
        with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as output:
            shutil.copyfileobj(response, output)
    assert asset_digest(destination) == EXPECTED_HASHES[name], f"Archive mismatch: {name}"
    checked_extract(destination, EXTRACT_ROOT / name)
    print("Verified:", destination.name)

# Notebook 20 checkpoints are supplied by you, not retrained.
g2_path = os.environ.get("CALORIE_G2_REPORT")
if g2_path:
    g2_path = Path(g2_path)
else:
    g2_path = DOWNLOAD_ROOT / "calorie_g2_pose_model_suite_report.zip"
    if not g2_path.exists() or sha256(g2_path) != EXPECTED_HASHES["g2_report"]:
        from google.colab import files
        print("Select calorie_g2_pose_model_suite_report.zip (the output of Notebook 20).")
        uploaded = files.upload()
        matches = [data for data in uploaded.values()
                   if hashlib.sha256(data).hexdigest() == EXPECTED_HASHES["g2_report"]]
        assert len(matches) == 1, "Please select the exact G2 report already analysed."
        g2_path.write_bytes(matches[0])
assert sha256(g2_path) == EXPECTED_HASHES["g2_report"], "G2 checkpoint archive mismatch"
checked_extract(g2_path, EXTRACT_ROOT / "g2_report")
print("All evidence verified. No raw videos or training downloads required.")


## Eğitimden dışlanmış kaynaklar ve kapsam kontrolü


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, column])
        for column in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


landmark_root = EXTRACT_ROOT / "calibration_landmarks"
manifest_path = next(landmark_root.rglob("manifest.csv"))
manifest = pd.read_csv(manifest_path).fillna("")
manifest["npz_path"] = manifest.output_path.map(
    lambda value: str(manifest_path.parent / value)
)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
    & manifest.group_id.isin(CALIBRATION_GROUPS)
].copy()
assert set(manifest.group_id) == CALIBRATION_GROUPS

windows, labels, groups, clip_ids, starts = [], [], [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        label = "unknown" if row.canonical_class in RETIRED_CLASSES else row.canonical_class
        labels.append(label)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
starts = np.asarray(starts)
y_gate = (labels != "unknown").astype(np.int64)

for version in ["v6c_report"]:
    split_path = EXTRACT_ROOT / version / "split_assignments.csv"
    split_groups = set(pd.read_csv(split_path).group_id)
    assert not (set(groups) & split_groups), f"Calibration leakage into {version}"

print("Calibration segments:", len(manifest))
print("Calibration windows:", len(X))
display(pd.crosstab(pd.Series(groups, name="creator"), pd.Series(labels, name="label")))

assert X.shape[1:] == (64, 17, 4) and np.isfinite(X).all()
assert len(set(zip(groups, clip_ids, starts))) == len(X), "Duplicate windows"
manifest.to_csv(REPORT_ROOT / "used_manifest.csv", index=False)
coverage = pd.crosstab(pd.Series(groups, name="creator"), pd.Series(labels, name="label"))
coverage.to_csv(REPORT_ROOT / "coverage_by_creator.csv")
print("WARNING: historical draft-label notes remain unresolved; all scores are provisional.")


## Sabit V6-C modeli


In [ ]:
BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10), (7, 5, 11), (8, 6, 12),
    (5, 11, 13), (6, 12, 14), (11, 13, 15), (12, 14, 16),
]


def features_v6b(sample):
    return sample[:, :, :2].reshape(WINDOW, -1).astype(np.float32)


def features_v6c(sample):
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    bones = np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1)
    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        a = xy[:, first] - xy[:, center]
        b = xy[:, last] - xy[:, center]
        denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
        cosine = np.sum(a * b, axis=1) / np.maximum(denominator, 1e-6)
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    return np.concatenate([
        xy.reshape(WINDOW, -1), velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1), np.stack(angles, axis=1),
    ], axis=1).astype(np.float32)


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


def load_system(version, feature_function):
    root = EXTRACT_ROOT / f"{version}_report"
    checkpoint_path = next(root.glob("two_stage_tcn_*.pt"))
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    supported = checkpoint["supported_classes"]
    gate = SmallTCN(checkpoint["input_features"], 2).to(DEVICE)
    exercise = SmallTCN(checkpoint["input_features"], len(supported)).to(DEVICE)
    gate.load_state_dict(checkpoint["gate_state_dict"])
    exercise.load_state_dict(checkpoint["exercise_state_dict"])
    gate.eval(); exercise.eval()
    features = np.stack([feature_function(window) for window in X]).transpose(0, 2, 1)
    gate_parts, exercise_parts = [], []
    with torch.no_grad():
        for start in range(0, len(features), 256):
            batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
            gate_parts.append(gate(batch).softmax(1).cpu().numpy())
            exercise_parts.append(exercise(batch).softmax(1).cpu().numpy())
    return checkpoint, np.concatenate(gate_parts), np.concatenate(exercise_parts)


v6c_checkpoint, v6c_gate, v6c_exercise = load_system("v6c", features_v6c)
SUPPORTED_CLASSES = list(v6c_checkpoint["supported_classes"])
ALL_CLASSES = SUPPORTED_CLASSES + ["unknown"]
assert set(labels) <= set(ALL_CLASSES)
print("Frozen classes:", SUPPORTED_CLASSES)


## Notebook 20 ile aynı model tanımları


In [ ]:
NODE_FEATURE_CONFIG = {
    "coordinates_xy_per_joint": 2,
    "velocity_xy_per_joint": 2,
}
NODE_FEATURES = sum(NODE_FEATURE_CONFIG.values())


def pose_to_graph_features(sample):
    """Convert [time, 17, 4] pose to [channels, time, joints]."""
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    features = np.concatenate([xy, velocity], axis=2).transpose(2, 0, 1)
    assert features.shape == (NODE_FEATURES, WINDOW, 17), features.shape
    assert np.isfinite(features).all()
    return features.astype(np.float32)



SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]


def normalized_adjacency(num_nodes=17):
    adjacency = np.eye(num_nodes, dtype=np.float32)
    for first, second in SKELETON_EDGES:
        adjacency[first, second] = adjacency[second, first] = 1.0
    degree = adjacency.sum(axis=1)
    inverse_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-6)))
    return (inverse_sqrt @ adjacency @ inverse_sqrt).astype(np.float32)


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, hidden=96):
        super().__init__()
        self.input_projection = nn.Sequential(nn.Linear(17 * NODE_FEATURES, 128), nn.LayerNorm(128), nn.ReLU())
        self.gru = nn.GRU(128, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.classifier = nn.Sequential(nn.LayerNorm(hidden * 4), nn.Dropout(0.25), nn.Linear(hidden * 4, num_classes))
    def forward(self, x):
        x = x.permute(0, 2, 3, 1).reshape(x.size(0), x.size(2), -1)
        x, _ = self.gru(self.input_projection(x))
        pooled = torch.cat([x.mean(1), x.amax(1)], dim=1)
        return self.classifier(pooled)


class PoseTransformerClassifier(nn.Module):
    def __init__(self, num_classes, width=128, heads=4, layers=3):
        super().__init__()
        self.input_projection = nn.Linear(17 * NODE_FEATURES, width)
        self.position = nn.Parameter(torch.zeros(1, WINDOW, width))
        layer = nn.TransformerEncoderLayer(width, heads, width * 3, dropout=0.2, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers)
        self.classifier = nn.Sequential(nn.LayerNorm(width * 2), nn.Dropout(0.25), nn.Linear(width * 2, num_classes))
    def forward(self, x):
        x = x.permute(0, 2, 3, 1).reshape(x.size(0), x.size(2), -1)
        x = self.encoder(self.input_projection(x) + self.position[:, :x.size(1)])
        return self.classifier(torch.cat([x.mean(1), x.amax(1)], dim=1))


class AdaptiveGraphBlock(nn.Module):
    def __init__(self, in_channels, out_channels, temporal_kernel=9, dropout=0.2):
        super().__init__()
        self.register_buffer("base_adjacency", torch.from_numpy(normalized_adjacency()))
        self.learned_adjacency = nn.Parameter(torch.zeros(17, 17))
        self.q = nn.Conv2d(in_channels, max(8, out_channels // 4), 1)
        self.k = nn.Conv2d(in_channels, max(8, out_channels // 4), 1)
        self.spatial = nn.Conv2d(in_channels, out_channels, 1)
        self.temporal = nn.Sequential(
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, (temporal_kernel, 1), padding=((temporal_kernel - 1)//2, 0)),
            nn.BatchNorm2d(out_channels), nn.Dropout(dropout),
        )
        self.residual = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    def forward(self, x):
        q = self.q(x).mean(2).transpose(1, 2)
        k = self.k(x).mean(2)
        dynamic = torch.softmax(torch.matmul(q, k) / np.sqrt(k.size(1)), dim=-1)
        adjacency = self.base_adjacency + torch.tanh(self.learned_adjacency)
        mixed = torch.einsum("bctv,vw->bctw", x, adjacency) + torch.einsum("bctv,bvw->bctw", x, dynamic)
        return torch.relu(self.temporal(self.spatial(mixed)) + self.residual(x))


class AdaptiveGCNClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        channels = [32, 64, 96, 128]
        blocks=[]; current=NODE_FEATURES
        for out in channels:
            blocks.append(AdaptiveGraphBlock(current, out)); current=out
        self.features=nn.Sequential(*blocks)
        self.classifier=nn.Linear(current, num_classes)
    def forward(self, x):
        return self.classifier(self.features(x).mean((2,3)))


MODEL_FACTORIES = {
    "bigru": lambda n: BiGRUClassifier(n),
    "pose_transformer": lambda n: PoseTransformerClassifier(n),
    "adaptive_gcn": lambda n: AdaptiveGCNClassifier(n),
}


## CPU üzerinde tahmin — eğitim yok


In [ ]:
probabilities = {"v6c": v6c_exercise}
inference_seconds = {}
graph_features = np.stack([pose_to_graph_features(window) for window in X])
for name, factory in MODEL_FACTORIES.items():
    # Checkpoints in this archive are tensor state dictionaries with simple metadata.
    checkpoint = torch.load(EXTRACT_ROOT / "g2_report" / f"{name}.pt",
                            map_location=DEVICE, weights_only=True)
    assert checkpoint["model"] == name
    assert checkpoint["supported_classes"] == SUPPORTED_CLASSES, "Class order mismatch"
    assert checkpoint["window_frames"] == WINDOW
    assert checkpoint["node_features"] == NODE_FEATURE_CONFIG
    model = factory(len(SUPPORTED_CLASSES)).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    parts = []
    begun = time.perf_counter()
    with torch.inference_mode():
        for start in range(0, len(X), 128):
            batch = torch.from_numpy(graph_features[start:start + 128]).to(DEVICE)
            parts.append(model(batch).softmax(1).cpu().numpy())
    probabilities[name] = np.concatenate(parts)
    inference_seconds[name] = time.perf_counter() - begun
    print(name, "finished:", round(inference_seconds[name], 1), "seconds")
probabilities["equal_ensemble"] = np.mean(
    np.stack([probabilities[name] for name in MODEL_FACTORIES]), axis=0)
for probability in probabilities.values():
    assert probability.shape == (len(X), len(SUPPORTED_CLASSES))
    assert np.isfinite(probability).all()
    assert np.allclose(probability.sum(axis=1), 1, atol=1e-5)


## Aynı örneklerde karşılaştır, kapsamı raporla, ZIP indir


In [ ]:
# One predefined threshold for all systems. No sweeps, fitting or test-based weights.
present_labels = [label for label in ALL_CLASSES if label in set(labels)]
missing_labels = [label for label in SUPPORTED_CLASSES if label not in set(labels)]
print("Supported classes missing from this provisional set:", missing_labels)
rows, creator_rows, prediction_frames, class_rows = [], [], [], []
final_predictions = {}
for name, probability in probabilities.items():
    exercise_prediction = np.asarray(SUPPORTED_CLASSES)[probability.argmax(axis=1)]
    final = np.where(v6c_gate[:, 1] >= FROZEN_GATE_THRESHOLD, exercise_prediction, "unknown")
    final_predictions[name] = final
    supported_mask = labels != "unknown"
    present_supported = [label for label in SUPPORTED_CLASSES if label in set(labels)]
    creator_scores = []
    for group in sorted(set(groups)):
        mask = groups == group
        group_labels = [label for label in ALL_CLASSES if label in set(labels[mask])]
        group_f1 = f1_score(labels[mask], final[mask], labels=group_labels,
                            average="macro", zero_division=0)
        creator_scores.append(group_f1)
        creator_rows.append({"model": name, "creator": group, "windows": int(mask.sum()),
                             "present_label_count": len(group_labels),
                             "macro_f1_present_labels": group_f1,
                             "accuracy": accuracy_score(labels[mask], final[mask])})
    row = {"model": name, "windows": len(X), "present_label_count": len(present_labels),
           "macro_f1_present_labels": f1_score(labels, final, labels=present_labels,
                                                average="macro", zero_division=0),
           "accuracy": accuracy_score(labels, final),
           "creator_mean_macro_f1_present_labels": float(np.mean(creator_scores)),
           "unknown_recall": float(np.mean(final[labels == "unknown"] == "unknown")),
           "oracle_exercise_macro_f1_present_labels": f1_score(
               labels[supported_mask], exercise_prediction[supported_mask],
               labels=present_supported, average="macro", zero_division=0)}
    rows.append(row)
    report = classification_report(labels, final, labels=ALL_CLASSES,
                                   output_dict=True, zero_division=0)
    (REPORT_ROOT / f"{name}_classification_report.json").write_text(json.dumps(report, indent=2))
    for label in ALL_CLASSES:
        class_rows.append({"model": name, "class": label, **report[label]})
    prediction_frames.append(pd.DataFrame({
        "model": name, "group_id": groups, "clip_id": clip_ids, "start_frame": starts,
        "true_class": labels, "gate_probability": v6c_gate[:, 1],
        "exercise_prediction": exercise_prediction, "predicted_class": final,
    }))
    matrix = confusion_matrix(labels, final, labels=ALL_CLASSES)
    plt.figure(figsize=(12, 10))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=ALL_CLASSES, yticklabels=ALL_CLASSES)
    plt.title(f"{name} — PROVISIONAL creator comparison; gate 0.675")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_confusion_matrix.png", dpi=130)
    plt.close()

# Compare against the same V6-C predictions, not its unrelated internal-test score.
comparison = pd.DataFrame(rows)
baseline_f1 = float(comparison.loc[comparison.model == "v6c", "macro_f1_present_labels"].iloc[0])
comparison["macro_f1_delta_vs_v6c_pp"] = (comparison.macro_f1_present_labels - baseline_f1) * 100
for prediction in final_predictions.values():
    assert np.array_equal(prediction == "unknown", final_predictions["v6c"] == "unknown")
display(comparison.round(4))
display(pd.DataFrame(creator_rows).round(4))
comparison.to_csv(REPORT_ROOT / "system_comparison.csv", index=False)
pd.DataFrame(creator_rows).to_csv(REPORT_ROOT / "creator_metrics.csv", index=False)
pd.DataFrame(class_rows).to_csv(REPORT_ROOT / "class_metrics.csv", index=False)
pd.concat(prediction_frames, ignore_index=True).to_csv(REPORT_ROOT / "window_predictions.csv", index=False)
np.savez_compressed(REPORT_ROOT / "class_probabilities.npz", classes=np.asarray(SUPPORTED_CLASSES),
                    true_labels=labels, groups=groups, clip_ids=clip_ids, starts=starts,
                    gate_probability=v6c_gate[:, 1], **probabilities)
summary = {
    "experiment": "g2_existing_creator_provisional_comparison",
    "evidence_status": EVIDENCE_STATUS, "promotion_allowed": False,
    "gate_threshold": FROZEN_GATE_THRESHOLD, "threshold_tuned_here": False,
    "trained_here": False, "creator_groups": sorted(set(groups)),
    "present_labels": present_labels, "missing_supported_classes": missing_labels,
    "models": comparison.to_dict(orient="records"),
    "input_digests": EXPECTED_HASHES, "digest_algorithms": {"v6c_report": "git_blob_sha1", "calibration_landmarks": "git_blob_sha1", "g2_report": "sha256"}, "repository_commit": PINNED_COMMIT,
    "inference_seconds": inference_seconds,
    "limitations": ["Existing calibration creators, not a fresh locked test",
                    "Historical draft-label concerns are unresolved",
                    "Incomplete exercise coverage; correlated windows are not independent people",
                    "No threshold or ensemble-weight tuning; no automatic model promotion",
                    "G2 split metadata was not exported; exclusion relies on pinned Notebook 20 protocol and V6 split audit",
                    "This ensemble score is computed from actual averaged predictions on this set; it does not repair Notebook 20 internal validation"],
}
(REPORT_ROOT / "metrics.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
(REPORT_ROOT / "README.txt").write_text(
    "PROVISIONAL comparison only. Read metrics.json limitations.\n"
    "Macro F1 is over labels present in this calibration set, not the 15-label internal test.\n"
    "Absent classes have support=0; their reported zeros are not measured performance.\n"
    "Unknown rejection is identical because the same frozen gate is used for every classifier.\n"
    "Return the ZIP to the chat; do not replace the main model automatically.\n")
zip_path = shutil.make_archive(str(WORK_ROOT / "calorie_g2_creator_comparison_report"),
                               "zip", root_dir=REPORT_ROOT)
print("Report ready:", zip_path)
if not os.environ.get("CALORIE_LOCAL_REPO"):
    from google.colab import files
    files.download(zip_path)
